In [1]:
!pip install safetensors

In [2]:
import os
import subprocess
from google.colab import drive, userdata

# Prevent PyTorch Memory Fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configuration for GitHub
REPO_DIR = '/content/drive/MyDrive/master_thesis_FM/anysat_pipeline' # Update to your local Drive repo path
BRANCH_NAME = 'main' # Update if using a different branch

def setup_git():
    if not os.path.exists(REPO_DIR):
        print(f"Directory {REPO_DIR} does not exist. Please create/sync it first.")
        return

    os.chdir(REPO_DIR)

    # Configure Git
    subprocess.run(['git', 'config', '--global', 'user.name', 'Colab User'])
    subprocess.run(['git', 'config', '--global', 'user.email', 'colab@example.com'])

    try:
        # Retrieve PAT from secrets
        GITHUB_PAT = userdata.get('GITHUB_PAT')

        # Ensure remote URL uses PAT for seamless pulling
        remote_url = subprocess.check_output(['git', 'config', '--get', 'remote.origin.url']).decode('utf-8').strip()
        if '@' not in remote_url and GITHUB_PAT:
            new_url = remote_url.replace('https://', f'https://{GITHUB_PAT}@')
            subprocess.run(['git', 'remote', 'set-url', 'origin', new_url])

        print(f"Pulling latest changes from branch: {BRANCH_NAME}")
        subprocess.run(['git', 'fetch', 'origin'], check=True)
        subprocess.run(['git', 'checkout', BRANCH_NAME], check=True)
        subprocess.run(['git', 'pull', 'origin', BRANCH_NAME], check=True)
        print("✅ Git sync complete.")

    except Exception as e:
        print(f"⚠️ Git Setup issue (Proceeding anyway): {e}")

setup_git()

In [3]:
from dataclasses import dataclass
import torch

@dataclass(frozen=True)
class PipelineConfig:
    # Environment & Reproducibility
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Google Drive Archive Paths (UPDATE THESE to your arbitrary folders)
    DRIVE_ARCHIVE_SAR_TRAIN: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz"
    DRIVE_ARCHIVE_SAR_VAL: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/val_20%_random_out_of_orig_train_partition.tar.gz"
    DRIVE_ARCHIVE_SAR_TEST: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/orig_test_partition.tar.gz"
    DRIVE_ARCHIVE_LBL_TRAIN: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz"
    DRIVE_ARCHIVE_LBL_VAL: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/val_20%_random_out_of_orig_train_partition.tar.gz"
    DRIVE_ARCHIVE_LBL_TEST: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/test.tar.gz"

    # Local Extraction Paths (Fast I/O)
    LOCAL_DATA_ROOT: str = "/content/local_data"

    # AnySat Params
    PATCH_SIZE: int = 40
    S1_DATE_DUMMY: int = 196

    # Training Hyperparameters
    EPOCHS: int = 50
    TRAIN_BATCH_SIZE: int = 8      # Linear layer can easily handle this
    INFERENCE_BATCH_SIZE: int = 2  # Safe size for AnySat forward passes (Val/Test/Cache)
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 0.05
    PATIENCE: int = 10  # Early stopping
    NUM_WORKERS: int = 4      # Safe baseline for heavy tensors
    PREFETCH_FACTOR: int = 1  # Limits how many batches each worker hoards


    # Output Folders on Drive
    EXP_NAME: str = f"AS_LP_OTF_RANDOM10%_train_v1_patSize{PATCH_SIZE}_BS{TRAIN_BATCH_SIZE}_LR{str(LEARNING_RATE).split('.')[1]}_WD{str(WEIGHT_DECAY).split('.')[1]}_EP{EPOCHS}_PAT{PATIENCE}_SEED{SEED}"
    OUTPUT_ROOT: str = f"/content/drive/MyDrive/master_thesis_FM/experiments/AnySat/linear_probing/dense_features/{EXP_NAME}"

CONFIG = PipelineConfig()
os.makedirs(CONFIG.OUTPUT_ROOT, exist_ok=True)

In [4]:
import random
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pandas as pd
import gc

def seed_everything(seed=CONFIG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"✅ Environment initialized. Using device: {CONFIG.DEVICE}")

✅ Environment initialized. Using device: cuda


In [5]:
import shutil
import tarfile

def process_archive(drive_path, category, split):
    extract_dir = os.path.join(CONFIG.LOCAL_DATA_ROOT, split, category)
    os.makedirs(extract_dir, exist_ok=True)

    local_tar_path = os.path.join(CONFIG.LOCAL_DATA_ROOT, f"temp_{category}_{split}.tar.gz")

    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 0:
        print(f"⏩ Data for {split}/{category} already extracted.")
        return extract_dir

    print(f"📦 Copying {drive_path} to local storage...")
    shutil.copy2(drive_path, local_tar_path)

    print(f"🔓 Extracting {local_tar_path}...")
    with tarfile.open(local_tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_dir)

    print(f"🗑️ Deleting local archive {local_tar_path} to free space...")
    os.remove(local_tar_path)

    return extract_dir

print("🚀 Starting Sequential Data Transfer...")
dir_sar_train = process_archive(CONFIG.DRIVE_ARCHIVE_SAR_TRAIN, "sar", "train")
dir_lbl_train = process_archive(CONFIG.DRIVE_ARCHIVE_LBL_TRAIN, "label", "train")

dir_sar_val = process_archive(CONFIG.DRIVE_ARCHIVE_SAR_VAL, "sar", "val")
dir_lbl_val = process_archive(CONFIG.DRIVE_ARCHIVE_LBL_VAL, "label", "val")

dir_sar_test = process_archive(CONFIG.DRIVE_ARCHIVE_SAR_TEST, "sar", "test")
dir_lbl_test = process_archive(CONFIG.DRIVE_ARCHIVE_LBL_TEST, "label", "test")
print("✅ Data transfer complete. SSD Spikes minimized.")

🚀 Starting Sequential Data Transfer...
📦 Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz to local storage...
🔓 Extracting /content/local_data/temp_sar_train.tar.gz...


/tmp/ipykernel_459/4053595078.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


🗑️ Deleting local archive /content/local_data/temp_sar_train.tar.gz to free space...
📦 Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz to local storage...
🔓 Extracting /content/local_data/temp_label_train.tar.gz...
🗑️ Deleting local archive /content/local_data/temp_label_train.tar.gz to free space...
📦 Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/val_20%_random_out_of_orig_train_partition.tar.gz to local storage...
🔓 Extracting /content/local_data/temp_sar_val.tar.gz...
🗑️ Deleting local archive /content/local_data/temp_sar_val.tar.gz to free space...
📦 Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/val_20%_random_out_of_orig_train_partition.tar.gz to

In [6]:
import re
import glob
from torch.utils.data import Dataset, DataLoader

class BioMasstersDataset(Dataset):
    def __init__(self, sar_dir, lbl_dir):
        self.sar_dir = sar_dir
        self.lbl_dir = lbl_dir
        self.pairs = self._map_files()

    def _map_files(self):
        sar_paths = glob.glob(os.path.join(self.sar_dir, "**", "*.pt"), recursive=True)
        lbl_paths = glob.glob(os.path.join(self.lbl_dir, "**", "*.tif"), recursive=True)

        lbl_dict = {re.search(r"([a-zA-Z0-9]{8})_agbm\.tif$", p).group(1): p for p in lbl_paths if re.search(r"([a-zA-Z0-9]{8})_agbm\.tif$", p)}

        pairs = []
        for sar_path in sar_paths:
            match = re.search(r"([a-zA-Z0-9]{8})_S1_10\.pt$", sar_path)
            if match and match.group(1) in lbl_dict:
                pairs.append({"id": match.group(1), "sar": sar_path, "lbl": lbl_dict[match.group(1)]})
        return sorted(pairs, key=lambda x: x["id"])

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        sar_tensor = torch.load(pair["sar"], weights_only=True).float()
        with rasterio.open(pair["lbl"]) as src:
            lbl_array = src.read(1)
        return sar_tensor, torch.from_numpy(lbl_array).float(), pair["id"]

raw_train_dataset = BioMasstersDataset(dir_sar_train, dir_lbl_train)

# Val and Test use the raw data and run through the encoder on-the-fly
val_loader = DataLoader(
    BioMasstersDataset(dir_sar_val, dir_lbl_val),
    batch_size=CONFIG.INFERENCE_BATCH_SIZE,
    shuffle=False,
    num_workers=CONFIG.NUM_WORKERS,     # ⬅️ Throttled
    prefetch_factor=CONFIG.PREFETCH_FACTOR, # ⬅️ Explicit limit
    pin_memory=True
)

test_loader = DataLoader(
    BioMasstersDataset(dir_sar_test, dir_lbl_test),
    batch_size=CONFIG.INFERENCE_BATCH_SIZE,
    shuffle=False,
    num_workers=CONFIG.NUM_WORKERS,     # ⬅️ Throttled
    prefetch_factor=CONFIG.PREFETCH_FACTOR, # ⬅️ Explicit limit
    pin_memory=True
)

In [7]:
import sys
from safetensors.torch import save_file

# Ensure AnySat repo is accessible
sys.path.append('/content/drive/MyDrive/master_thesis_FM/models/1_AnySat/feature_extraction/')
from hubconf import AnySat

CACHE_DIR = os.path.join(CONFIG.LOCAL_DATA_ROOT, "train_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Throttled loader to prevent Colab Swap File / RAM bloat
cache_loader = DataLoader(raw_train_dataset, batch_size=CONFIG.INFERENCE_BATCH_SIZE, shuffle=False, num_workers=1)

def build_training_cache(encoder, loader, cache_dir):
    print(f"⚙️ Building local training cache at {cache_dir}...")
    encoder.eval()

    with torch.no_grad():
        for sar, lbl, ids in tqdm(loader, desc="Caching Features"):
            sar = sar.to(CONFIG.DEVICE, non_blocking=True)
            dates = torch.tensor([CONFIG.S1_DATE_DUMMY] * sar.shape[0]).to(CONFIG.DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda'):
                data = {"s1": sar.unsqueeze(1), "s1_dates": dates}
                features = encoder(data, patch_size=CONFIG.PATCH_SIZE, output='dense', output_modality='s1')
                if features.shape[1] == 1536:
                    features = features.permute(0, 2, 3, 1)

            # Move to CPU and explicitly force the entire batch to be contiguous
            features = features.cpu().to(torch.float16).contiguous()
            lbl = lbl.cpu().contiguous()

            for i, file_id in enumerate(ids):
                cache_path = os.path.join(cache_dir, f"{file_id}.safetensors")
                if not os.path.exists(cache_path):
                    # .clone() ensures memory isolation
                    # .contiguous() strictly satisfies the safetensors byte-writer
                    save_file({
                        "features": features[i].clone().contiguous(),
                        "label": lbl[i].clone().contiguous()
                    }, cache_path)

            del sar, lbl, dates, data, features
            torch.cuda.empty_cache()
            gc.collect()

temp_encoder = AnySat.from_pretrained('base', flash_attn=False).to(CONFIG.DEVICE)
build_training_cache(temp_encoder, cache_loader, CACHE_DIR)

del temp_encoder
torch.cuda.empty_cache()
gc.collect()

Downloading: "https://huggingface.co/g-astruc/AnySat/resolve/main/models/AnySat.pth" to /root/.cache/torch/hub/checkpoints/AnySat.pth


100%|██████████| 480M/480M [00:01<00:00, 318MB/s]


⚙️ Building local training cache at /content/local_data/train_cache...


Caching Features:   0%|          | 0/87 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


17

In [8]:
import torch.nn as nn
from safetensors.torch import load_file
import glob

class CachedTrainDataset(Dataset):
    def __init__(self, cache_dir):
        # Read the new format
        self.files = glob.glob(os.path.join(cache_dir, "*.safetensors"))

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        # Instant, zero-copy reading
        data = load_file(self.files[idx])
        return data["features"].float(), data["label"].float()

# Max workers and Target Batch Size for ultra-fast training
cached_train_loader = DataLoader(
    CachedTrainDataset(CACHE_DIR),
    batch_size=CONFIG.TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=CONFIG.NUM_WORKERS,     # ⬅️ Throttled
    prefetch_factor=CONFIG.PREFETCH_FACTOR, # ⬅️ Explicit limit
    pin_memory=True
)

class HybridLinearProbing(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AnySat.from_pretrained('base', flash_attn=False)
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.regressor = nn.Linear(1536, 1)

    def forward(self, x, dates=None, use_cached=False):
        if use_cached:
            features = x
        else:
            with torch.no_grad():
                data = {"s1": x.unsqueeze(1), "s1_dates": dates}
                features = self.encoder(data, patch_size=CONFIG.PATCH_SIZE, output='dense', output_modality='s1')
                if features.shape[1] == 1536:
                    features = features.permute(0, 2, 3, 1)

        return self.regressor(features).squeeze(-1)

model = HybridLinearProbing().to(CONFIG.DEVICE)
model = torch.compile(model) # JIT compilation for extra speed
print("✅ Hybrid Model compiled.")

✅ Hybrid Model compiled.


In [9]:
class MetricAccumulator:
    def __init__(self): self.reset()
    def reset(self):
        self.sum_sq_error = self.sum_abs_error = self.sum_target = self.sum_target_sq = self.n_pixels = 0.0

    def update(self, preds, targets):
        preds, targets = preds.detach().cpu().float(), targets.detach().cpu().float()
        mask = ~torch.isnan(targets)
        if not mask.any(): return

        p, t = preds[mask], targets[mask]
        n = p.numel()

        self.sum_sq_error += torch.sum((p - t) ** 2).item()
        self.sum_abs_error += torch.sum(torch.abs(p - t)).item()
        self.sum_target += torch.sum(t).item()
        self.sum_target_sq += torch.sum(t ** 2).item()
        self.n_pixels += n

    def compute(self):
        if self.n_pixels == 0: return {"MSE": 0, "RMSE": 0, "MAE": 0, "R2": 0}
        mse = self.sum_sq_error / self.n_pixels
        mean_t = self.sum_target / self.n_pixels
        ss_tot = self.sum_target_sq - (self.sum_target ** 2) / self.n_pixels

        return {
            "MSE": mse, "RMSE": np.sqrt(mse), "MAE": self.sum_abs_error / self.n_pixels,
            "R2": 1.0 - (self.sum_sq_error / ss_tot) if ss_tot != 0 else 0.0
        }

def save_checkpoint(state, is_best):
    save_path = os.path.join(CONFIG.OUTPUT_ROOT, "checkpoint.pth")
    torch.save(state, save_path)
    if is_best: shutil.copyfile(save_path, os.path.join(CONFIG.OUTPUT_ROOT, "best_model.pth"))

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.regressor.parameters(), lr=CONFIG.LEARNING_RATE, weight_decay=CONFIG.WEIGHT_DECAY)
scaler = torch.amp.GradScaler()

start_epoch, best_val_loss, patience_counter, history = 0, float('inf'), 0, []

# Resume Logic
ckpt_path = os.path.join(CONFIG.OUTPUT_ROOT, "checkpoint.pth")
if os.path.exists(ckpt_path):
    print(f"🔄 Resuming from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch, best_val_loss = checkpoint['epoch'] + 1, checkpoint['best_val_loss']
    history = pd.read_csv(os.path.join(CONFIG.OUTPUT_ROOT, "training_history.csv")).to_dict('records')

for epoch in range(start_epoch, CONFIG.EPOCHS):

    # --- TRAIN PHASE (Cached Safetensors) ---
    model.train()
    train_metrics, train_loss_acc = MetricAccumulator(), 0.0

    for features, lbl in tqdm(cached_train_loader, desc=f"Epoch {epoch+1} [Train]"):
        features, lbl = features.to(CONFIG.DEVICE, non_blocking=True), lbl.to(CONFIG.DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            preds = model(features, use_cached=True)
            loss = criterion(preds, lbl)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_acc += loss.item()
        train_metrics.update(preds, lbl)
        del features, lbl, preds, loss

    train_res = train_metrics.compute()
    train_res["Loss"] = train_loss_acc / len(cached_train_loader)


    # --- VALIDATION PHASE (On-The-Fly) ---
    model.eval()
    val_metrics, val_loss_acc = MetricAccumulator(), 0.0

    with torch.no_grad():
        for sar, lbl, _ in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            sar, lbl = sar.to(CONFIG.DEVICE, non_blocking=True), lbl.to(CONFIG.DEVICE, non_blocking=True)
            dates = torch.tensor([CONFIG.S1_DATE_DUMMY] * sar.shape[0]).to(CONFIG.DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda'):
                preds = model(sar, dates, use_cached=False)
                loss = criterion(preds, lbl)

            val_loss_acc += loss.item()
            val_metrics.update(preds, lbl)
            del sar, lbl, dates, preds, loss

    val_res = val_metrics.compute()
    val_res["Loss"] = val_loss_acc / len(val_loader)


    # --- LOGGING & CHECKPOINTING ---
    history.append({
        "Epoch": epoch + 1, "Train_MSE": train_res["MSE"], "Val_MSE": val_res["MSE"],
        "Val_RMSE": val_res["RMSE"], "Val_MAE": val_res["MAE"], "Val_R2": val_res["R2"]
    })
    pd.DataFrame(history).to_csv(os.path.join(CONFIG.OUTPUT_ROOT, "training_history.csv"), index=False)

    print(f"📊 Epoch {epoch+1}: Train MSE: {train_res['MSE']:.4f} | Val MSE: {val_res['MSE']:.4f} | Val R2: {val_res['R2']:.4f}")

    is_best = val_res["MSE"] < best_val_loss
    if is_best:
        best_val_loss, patience_counter = val_res["MSE"], 0
    else:
        patience_counter += 1

    save_checkpoint({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scaler_state_dict': scaler.state_dict(), 'best_val_loss': best_val_loss}, is_best)

    # Force RAM cleanup to prevent OS OOM killer
    gc.collect()

    if patience_counter >= CONFIG.PATIENCE:
        print("🛑 Early stopping triggered.")
        break

Epoch 1 [Train]:   0%|          | 0/87 [00:00<?, ?it/s]

W0609 09:14:26.778000 459 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1 [Val]:   0%|          | 0/218 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

📊 Epoch 1: Train MSE: 9209.0724 | Val MSE: 9010.8990 | Val R2: -0.7152


...

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

📊 Epoch 10: Train MSE: 6428.3087 | Val MSE: 6356.6766 | Val R2: -0.2100


Epoch 11 [Train]:   0%|          | 0/87 [00:00<?, ?it/s]

Epoch 11 [Val]:   0%|          | 0/218 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

In [ ]:
df = pd.read_csv(os.path.join(CONFIG.OUTPUT_ROOT, "training_history.csv"))
plt.figure(figsize=(10, 6))
plt.plot(df['Epoch'], df['Train_MSE'], label='Train MSE', marker='o')
plt.plot(df['Epoch'], df['Val_MSE'], label='Validation MSE', marker='s')
plt.title('Training and Validation MSE')
plt.xlabel('Epoch'), plt.ylabel('MSE'), plt.legend(), plt.grid(True)
plt.savefig(os.path.join(CONFIG.OUTPUT_ROOT, "training_curve.png"))
plt.show()

In [ ]:
inference_model = HybridLinearProbing().to(CONFIG.DEVICE)
best_ckpt = torch.load(os.path.join(CONFIG.OUTPUT_ROOT, "best_model.pth"))
inference_model.load_state_dict(best_ckpt['model_state_dict'])
inference_model.eval()

test_metrics = MetricAccumulator()
print("🔍 Running Test Set...")
with torch.no_grad():
    for sar, lbl, _ in tqdm(test_loader, desc="Testing"):
        sar, lbl = sar.to(CONFIG.DEVICE, non_blocking=True), lbl.to(CONFIG.DEVICE, non_blocking=True)
        dates = torch.tensor([CONFIG.S1_DATE_DUMMY] * sar.shape[0]).to(CONFIG.DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            test_metrics.update(inference_model(sar, dates), lbl)

test_res = test_metrics.compute()
with open(os.path.join(CONFIG.OUTPUT_ROOT, "test_metrics_report.txt"), "w") as f:
    f.write(f"Best Val MSE: {best_ckpt['best_val_loss']:.4f}\n--- TEST ---\nMSE: {test_res['MSE']:.4f}\nRMSE: {test_res['RMSE']:.4f}\nMAE: {test_res['MAE']:.4f}\nR2: {test_res['R2']:.4f}\n")
print("✅ Evaluation complete.")

In [ ]:
def visualize(model, loader, num=3):
    model.eval()
    plt.figure(figsize=(12, 5 * num))
    plotted = 0
    with torch.no_grad():
        for sar, lbl, ids in loader:
            sar = sar.to(CONFIG.DEVICE)
            dates = torch.tensor([CONFIG.S1_DATE_DUMMY] * sar.shape[0]).to(CONFIG.DEVICE)
            with torch.amp.autocast('cuda'):
                preds = model(sar, dates).cpu().numpy()
            lbl = lbl.numpy()

            for i in range(sar.shape[0]):
                if plotted >= num: break
                plt.subplot(num, 2, plotted * 2 + 1)
                plt.imshow(lbl[i], cmap='viridis'), plt.title(f"Target | {ids[i]}"), plt.axis('off')
                plt.subplot(num, 2, plotted * 2 + 2)
                plt.imshow(preds[i], cmap='viridis'), plt.title(f"Pred | {ids[i]}"), plt.axis('off')
                plotted += 1
            if plotted >= num: break
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.OUTPUT_ROOT, "visual_comparison.png"))
    plt.show()

visualize(inference_model, test_loader, num=4)

In [ ]:
def visualize(model, loader, num=3):
    model.eval()
    plt.figure(figsize=(12, 5 * num))
    plotted = 0
    with torch.no_grad():
        for sar, lbl, ids in loader:
            sar = sar.to(CONFIG.DEVICE)
            dates = torch.tensor([CONFIG.S1_DATE_DUMMY] * sar.shape[0]).to(CONFIG.DEVICE)
            with torch.amp.autocast('cuda'):
                preds = model(sar, dates).cpu().numpy()
            lbl = lbl.numpy()

            for i in range(sar.shape[0]):
                if plotted >= num: break
                plt.subplot(num, 2, plotted * 2 + 1)
                plt.imshow(lbl[i], cmap='viridis'), plt.title(f"Target | {ids[i]}"), plt.axis('off')
                plt.subplot(num, 2, plotted * 2 + 2)
                plt.imshow(preds[i], cmap='viridis'), plt.title(f"Pred | {ids[i]}"), plt.axis('off')
                plotted += 1
            if plotted >= num: break
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.OUTPUT_ROOT, "visual_comparison.png"))
    plt.show()

visualize(inference_model, test_loader, num=4)

In [ ]:
from google.colab import runtime
print("💾 Flushing Drive...")
drive.flush_and_unmount()
print("👋 Terminating runtime.")
runtime.unassign()